<a href="https://colab.research.google.com/github/archanasathyan8762-afk/sentiment-analysis-reviews/blob/main/sentimentanaysis" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import pandas as pd
df=pd.read_csv("IMDB Dataset.csv")
print(df.shape)
print(df['sentiment'].value_counts())
print(df['review'][0][:200])

(50000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo


In [23]:
import re
def clean(text):
  text=text.lower()
  text=re.sub(r"<.*?>"," ",text)
  text=re.sub(r"[^a-z\s]","",text)
  text=re.sub(r"\s+"," ",text).strip()
  return text
sample="This movie is great!!!!<br/><br/>Loved it,10/10."
print(clean(sample))

this movie is great loved it


In [24]:
df['clean_review']=df['review'].apply(clean)
print(df['review'][0][:100])
print(df['clean_review'][0][:100])

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. The
one of the other reviewers has mentioned that after watching just oz episode youll be hooked they ar


In [25]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(len(X_train), len(X_test))

tfidf = TfidfVectorizer(max_features=20000,ngram_range=(1,2))
X_train_vec = tfidf.fit_transform(X_train)   # learns vocab from train ONLY
X_test_vec = tfidf.transform(X_test)         # just applies it

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

print("Accuracy:", accuracy_score(y_test, model.predict(X_test_vec)))


40000 10000
Accuracy: 0.9049


In [26]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test_vec)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

    negative       0.91      0.90      0.90      5000
    positive       0.90      0.91      0.91      5000

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000

[[4491  509]
 [ 442 4558]]


In [27]:
tests=["this movie was absolutely fantastic ,loved every minute","worst film ever ,a complete waste of time","it was not good at all"]
for t in tests:
  vec=tfidf.transform([clean(t)])
  print(t,"->",model.predict(vec)[0])

this movie was absolutely fantastic ,loved every minute -> positive
worst film ever ,a complete waste of time -> negative
it was not good at all -> negative


In [28]:
def predict_sentiment(review):
    cleaned = clean(review)                      # same cleaning as training
    vec = tfidf.transform([cleaned])             # same vocabulary as training
    label = model.predict(vec)[0]
    confidence = model.predict_proba(vec).max()  # highest class probability
    return label, round(confidence * 100, 1)

print(predict_sentiment("It was not good at all"))
print(predict_sentiment("Brilliant acting and a beautiful story"))

('negative', np.float64(92.4))
('positive', np.float64(95.8))
